# Clase 095 — K-Means: selección de K y MiniBatch

Segmentar datos no etiquetados con K-Means, elegir `K` con criterios reproducibles (elbow + silhouette) y escalar con `MiniBatchKMeans`.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. K-Means sobre blobs

`make_blobs` con 5 centros; ajustamos `KMeans` con `n_init` y `random_state` fijos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

np.random.seed(42)
X, y_true = make_blobs(n_samples=2000, centers=5, cluster_std=0.5, random_state=42)

km = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X)
print("inercia:", round(km.inertia_, 2), "| centroides:", km.cluster_centers_.shape)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap="tab10", s=8, alpha=0.6)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c="black", marker="X", s=150, label="centroides")
plt.legend(); plt.title("K-Means (K=5) sobre make_blobs")
plt.tight_layout(); plt.show()

## 2. Elbow method: inercia vs K

La inercia siempre baja al subir `K`; buscamos el "codo" donde la mejora se aplana.

In [ ]:
Ks = range(2, 11)
inercias = []
for k in Ks:
    inercias.append(KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_)

for k, i in zip(Ks, inercias):
    print(f"K={k}: inercia={i:.1f}")

plt.figure(figsize=(7, 4))
plt.plot(list(Ks), inercias, "o-", color="#37a")
plt.axvline(5, ls="--", color="#c33", lw=0.8, label="codo esperado (K=5)")
plt.xlabel("K"); plt.ylabel("inercia")
plt.title("Elbow method")
plt.legend(); plt.tight_layout(); plt.show()

## 3. Silhouette: elegir K objetivamente

El `silhouette_score` (en `[-1, 1]`) combina cohesión y separación. Su máximo indica el `K` óptimo.

In [ ]:
from sklearn.metrics import silhouette_score

sils = []
for k in Ks:
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    sils.append(silhouette_score(X, labels))

k_opt = list(Ks)[int(np.argmax(sils))]
for k, s in zip(Ks, sils):
    print(f"K={k}: silhouette={s:.4f}")
print("K optimo por silhouette:", k_opt)
assert k_opt == 5, "el silhouette deberia elegir K=5"

plt.figure(figsize=(7, 4))
plt.plot(list(Ks), sils, "o-", color="#3a7")
plt.axvline(k_opt, ls="--", color="#c33", lw=0.8, label=f"K optimo = {k_opt}")
plt.xlabel("K"); plt.ylabel("silhouette score")
plt.title("Silhouette vs K")
plt.legend(); plt.tight_layout(); plt.show()

## 4. El escalado importa

Con una feature 100x mayor, K-Means la deja dominar la distancia. `StandardScaler` corrige.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

X_skew = X.copy()
X_skew[:, 1] *= 100  # una feature domina

lab_sin = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(X_skew)
lab_con = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(
    StandardScaler().fit_transform(X_skew))

ari_sin = adjusted_rand_score(y_true, lab_sin)
ari_con = adjusted_rand_score(y_true, lab_con)
print(f"ARI sin escalar : {ari_sin:.4f}")
print(f"ARI con escalar : {ari_con:.4f}")
assert ari_con > ari_sin, "escalar deberia mejorar el clustering"
print("StandardScaler evita que una feature domine la distancia.")

## 5. MiniBatchKMeans: velocidad vs calidad

Sobre `load_digits`, `MiniBatchKMeans` es más rápido con inercia levemente peor.

In [ ]:
import time
from sklearn.datasets import load_digits
from sklearn.cluster import MiniBatchKMeans

Xd = load_digits().data

t0 = time.perf_counter()
km_full = KMeans(n_clusters=10, n_init=10, random_state=42).fit(Xd)
t_full = time.perf_counter() - t0

t0 = time.perf_counter()
km_mb = MiniBatchKMeans(n_clusters=10, batch_size=256, n_init=10, random_state=42).fit(Xd)
t_mb = time.perf_counter() - t0

print(f"KMeans      : {t_full*1000:6.1f} ms | inercia {km_full.inertia_:.0f}")
print(f"MiniBatch   : {t_mb*1000:6.1f} ms | inercia {km_mb.inertia_:.0f}")
print("MiniBatch cambia velocidad por un poco de inercia.")

## 6. K-Means falla en `make_moons`

K-Means asume clusters convexos: parte las lunas por la mitad. Para formas arbitrarias se usa DBSCAN (clase 096).

In [ ]:
from sklearn.datasets import make_moons

Xm, ym = make_moons(n_samples=500, noise=0.05, random_state=42)
lab_m = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(Xm)

plt.figure(figsize=(7, 5))
plt.scatter(Xm[:, 0], Xm[:, 1], c=lab_m, cmap="coolwarm", s=12)
plt.title("K-Means parte las lunas mal (usar DBSCAN)")
plt.tight_layout(); plt.show()
print("ARI vs verdad:", round(adjusted_rand_score(ym, lab_m), 4), "(bajo: K-Means no sirve aca)")

## Ejercicios

1. Ajustá `KMeans(n_clusters=5)` sobre blobs y graficá puntos + `cluster_centers_`.
2. Calculá `inertia_` y `silhouette_score` para `K = 2..10` y justificá el `K` elegido.
3. Repetí el ajuste sin escalar una feature 100x mayor y comparalo con `StandardScaler` previo.
4. Sobre `load_digits`, cronometrá `KMeans` vs `MiniBatchKMeans` y compará inercias.
5. Corré K-Means sobre `make_moons` y explicá visualmente por qué falla.

## Conclusiones

- **Escalá siempre** antes de K-Means: usa distancia euclídea y una feature grande domina.
- Fijá `n_init >= 10` y `random_state` para reproducibilidad (K-Means depende de la inicialización).
- Elegí `K` combinando elbow (inercia) + silhouette; priorizá silhouette si no hay codo claro.
- `MiniBatchKMeans` acelera a costa de un poco de inercia; K-Means no sirve para clusters no convexos.

## ✅ Soluciones de los ejercicios

Cinco ejercicios de K-Means: ajuste y centroides, selección de K, importancia del escalado, MiniBatch y los límites de K-Means en clusters no convexos. `n_jobs=1`.

**Ejercicio 1 — Blobs y centroides.** `KMeans(n_clusters=5)`; graficamos puntos por etiqueta y los `cluster_centers_`.

In [ ]:
import numpy as np, time, matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_digits, make_moons
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

X, ytrue = make_blobs(n_samples=800, centers=5, random_state=42)
km = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X)
plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap='tab10', s=10)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='k', marker='X', s=150)
plt.title('K-Means k=5 + centroides'); plt.tight_layout(); plt.show()
print('inercia:', round(km.inertia_, 1))

**Ejercicio 2 — Elegir K.** Curvas de inercia (codo) y silhouette.

In [ ]:
Ks = range(2, 11)
inertias, sils = [], []
for k in Ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(m.inertia_)
    sils.append(silhouette_score(X, m.labels_))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(list(Ks), inertias, 'o-'); ax[0].set_title('Codo (inercia)'); ax[0].set_xlabel('K')
ax[1].plot(list(Ks), sils, 'o-'); ax[1].set_title('Silhouette'); ax[1].set_xlabel('K')
plt.tight_layout(); plt.show()
best_k = list(Ks)[int(np.argmax(sils))]
print('K optimo por silhouette:', best_k)
assert best_k in (4, 5), 'silhouette deberia sugerir 4-5 en 5 blobs con solape'
print('nota: con estos blobs dos grupos se solapan, por eso silhouette puede marcar 4')

**Ejercicio 3 — El escalado importa.** Con una feature 100× mayor, K-Means la deja dominar la distancia; `StandardScaler` lo corrige.

In [ ]:
rng = np.random.default_rng(0)
Xr, _ = make_blobs(n_samples=600, centers=3, random_state=1)
Xr[:, 1] = Xr[:, 1] * 100  # una feature con escala 100x
lab_raw = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(Xr)
lab_scaled = KMeans(n_clusters=3, n_init=10,
                    random_state=42).fit_predict(StandardScaler().fit_transform(Xr))
sil_raw = silhouette_score(Xr, lab_raw)
sil_scaled = silhouette_score(StandardScaler().fit_transform(Xr), lab_scaled)
print(f'silhouette sin escalar: {sil_raw:.3f}')
print(f'silhouette escalado   : {sil_scaled:.3f}')
print('Sin escalar, la feature grande manda; escalar equilibra las distancias.')

**Ejercicio 4 — MiniBatch vs full.** Sobre `load_digits`, comparamos tiempo e inercia.

In [ ]:
Xd, _ = load_digits(return_X_y=True)
def timed(model):
    t0 = time.perf_counter(); m = model.fit(Xd); return time.perf_counter() - t0, m.inertia_
t_km, in_km = timed(KMeans(n_clusters=10, n_init=10, random_state=42))
t_mb, in_mb = timed(MiniBatchKMeans(n_clusters=10, batch_size=256, n_init=10, random_state=42))
print(f'KMeans      : {t_km*1000:6.1f} ms | inercia {in_km:,.0f}')
print(f'MiniBatch   : {t_mb*1000:6.1f} ms | inercia {in_mb:,.0f}')
print('MiniBatch cambia un poco de inercia por mucha velocidad: ideal para datos grandes.')

**Ejercicio 5 — K-Means falla en `make_moons`.** Asume clusters esféricos; parte las lunas. La respuesta es DBSCAN (clase 096).

In [ ]:
Xm, ym = make_moons(n_samples=500, noise=0.05, random_state=42)
lab = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(Xm)
plt.figure(figsize=(6, 4))
plt.scatter(Xm[:, 0], Xm[:, 1], c=lab, cmap='coolwarm', s=10)
plt.title('K-Means parte las lunas por la mitad'); plt.tight_layout(); plt.show()
print('K-Means asume grupos convexos/esfericos -> falla en formas no convexas.')
print('Solucion: clustering por densidad (DBSCAN), que veremos en la clase 096.')